# GloVe, TF-IDF weighted mean, concatenated + 2-layer MLP

**Task:** Abductive Commonsense Reasoning (ART). Given two observations O1, O2 and two
hypotheses H1, H2, pick the hypothesis that better explains what happened in between.

**Project constraint:** static, pretrained word embeddings that are never trained, and a
classifier with 2 layers and a ReLU non-linearity. Only the classifier is trained.

**Approach of this notebook.** Same pipeline as `../glove_concatenate_bigger_mlp.ipynb`, with these changes:

- the sentence vector is the **TF-IDF weighted mean** of the GloVe word vectors instead of the plain mean, so frequent words like "the", "was" and "he" count less
- the classifier is reduced to the 2 layers the project asks for: `Linear(1200, 256) → ReLU → Linear(256, 2)` since this is a project constrainted

### What the earlier static runs taught me

- **Averaging the four sentences cannot work.** The mean of O1, O2, H1, H2 does not change when H1 and H2 swap, so the loss stays at ln 2. Concatenating them in a fixed order fixes that.
- **Validation and test disagree.** A word-overlap rule without any model reaches 63.5% on validation and 51.9% on test. The models learn this shortcut. Test is very likely the adversarially filtered ART dev set.

In [ ]:
%pip install -q datasets gensim lightning nltk python-dotenv scikit-learn wandb

In [1]:
import os
import warnings
from collections import Counter
from pathlib import Path

import gensim.downloader as api
import lightning as L
import nltk
import numpy as np
import torch
import torch.nn as nn
import wandb
from datasets import load_dataset
from dotenv import find_dotenv, load_dotenv
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from lightning.pytorch.loggers import WandbLogger
from sklearn.feature_extraction.text import TfidfVectorizer
from torch.utils.data import DataLoader, TensorDataset

nltk.download("punkt_tab", quiet=True)

True

In [2]:
dotenv_path = find_dotenv(usecwd=True)
load_dotenv(dotenv_path)
print(f"environment loaded from: {dotenv_path or '<no .env found, using the defaults below>'}")

WANDB_BASE_URL = os.getenv("WANDB_BASE_URL", "https://api.wandb.ai")
WANDB_ENTITY = os.getenv("WANDB_ENTITY", "cmedi32-hochschule-luzern")
WANDB_PROJECT = os.getenv("WANDB_PROJECT", "nlp-project-01-abductive-reasoning")

os.environ["WANDB_BASE_URL"] = WANDB_BASE_URL
os.environ["WANDB_ENTITY"] = WANDB_ENTITY
os.environ["WANDB_PROJECT"] = WANDB_PROJECT

wandb.login()

environment loaded from: /home/cedi/01_Projects/nlp-project-01-abductive-reasoning/.env


wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: cmedi32 (cmedi32-hochschule-luzern) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [3]:
config = {
    "dataset": "amarfurt/art-nlp-hs26",
    "tokenizer": "nltk.word_tokenize",
    "lowercase": True,
    "remove_punctuation": True,
    "remove_stopwords": False,  # tf-idf gives frequent words a low weight instead of dropping them
    "normalization": "none",
    "truncation": "none",
    "embedding": "glove-wiki-gigaword-300",
    "embedding_trainable": False,  # static: vectors are only looked up, never updated
    "sentence_representation": "weighted_mean",
    "sentence_weighting": "tfidf",  # "tfidf" or "none" (plain mean, for the ablation)
    "tfidf_fit_on": "train sentences (O1, O2, H1, H2)",
    "example_representation": "concat",  # O1, O2, H1, H2
    "classifier": "mlp",  # linear -> relu -> linear
    "hidden_dim": 256,
    "learning_rate": 1e-3,
    "batch_size": 64,
    "max_epochs": 100,
    "patience": 5,  # validation checks without improvement of loss/valid before stopping
    "seed": 42,
}

# one name per variant, so the ablation does not overwrite the checkpoint of the main run
MODEL_NAME = {"tfidf": "glove_tfidf_mlp", "none": "glove_mean_mlp"}[config["sentence_weighting"]]

run = wandb.init(
    entity=WANDB_ENTITY,
    project=WANDB_PROJECT,
    name=MODEL_NAME.replace("_", "-"),
    config=config,
)

In [4]:
dataset = load_dataset("amarfurt/art-nlp-hs26")

train = dataset["train"]
valid = dataset["validation"]
test = dataset["test"]

print(f"train: {len(train)}, valid: {len(valid)}, test: {len(test)}")

train: 16801, valid: 1000, test: 1532


### Preprocessing

In [5]:
def preprocess(text):
    tokens = nltk.word_tokenize(text)
    return [token.lower() for token in tokens if any(character.isalnum() for character in token)] # @todo decisions here have to be grounded in the vector lookup table

In [6]:
TEXT_FIELDS = ["observation_1", "observation_2", "hypothesis_1", "hypothesis_2"]

def preprocess_example(example):
    processed = {field: preprocess(example[field]) for field in TEXT_FIELDS}
    processed["target"] = example["label"] - 1
    return processed


def preprocess_split(split):
    return split.map(preprocess_example)

In [7]:
train_tokens = preprocess_split(train)
valid_tokens = preprocess_split(valid)
test_tokens = preprocess_split(test)

### Static word embeddings (frozen)

GloVe is loaded once and only used as a lookup table. The word vectors are turned into numpy
features before training starts, so no gradient ever reaches them. The only trainable weights
in this notebook are the two linear layers of the MLP below.

In [13]:
wv = api.load(config["embedding"])
print(f"vocabulary: {len(wv.index_to_key)} words, {wv.vector_size} dimensions")

vocabulary: 400000 words, 300 dimensions


### TF-IDF weighting

In [14]:
# documents for the idf: every single training sentence (O1, O2, H1, H2), already tokenized
train_sentences = [example[field] for example in train_tokens for field in TEXT_FIELDS]

# the sentences are already token lists, so the analyzer hands them through unchanged
vectorizer = TfidfVectorizer(analyzer=lambda tokens: tokens)
vectorizer.fit(train_sentences)

idf = dict(zip(vectorizer.get_feature_names_out(), vectorizer.idf_))
idf_unseen = np.log(1 + len(train_sentences)) + 1  # df = 0 in the smoothed formula -> highest possible idf

print(f"tf-idf fitted on {len(train_sentences)} training sentences, vocabulary: {len(idf)} terms")
print(f"idf range: {vectorizer.idf_.min():.2f} – {vectorizer.idf_.max():.2f}, words never seen in train: {idf_unseen:.2f}")

tf-idf fitted on 67204 training sentences, vocabulary: 16189 terms
idf range: 2.06 – 11.42, words never seen in train: 12.12


In [16]:
# @todo use this cell for the presentation: which words lose weight, and how does one sentence change # 
# @todo this is unpresentable as a python beginner, simplify it
lowest = sorted(idf.items(), key=lambda item: item[1])[:15]
print("lowest idf:", ", ".join(f"{word} {weight:.2f}" for word, weight in lowest))
print()

example = train_tokens[0]["hypothesis_1"]
weights = np.array([idf.get(token, idf_unseen) for token in example])
print(f"{'token':<15}{'plain mean':>12}{'tf-idf':>10}")
for token, share in zip(example, weights / weights.sum()):
    print(f"{token:<15}{1 / len(example):>12.1%}{share:>10.1%}")

lowest idf: the 2.06, to 2.20, a 2.24, was 2.47, and 2.66, her 3.07, his 3.17, he 3.20, i 3.28, in 3.33, she 3.39, for 3.43, of 3.49, had 3.50, it 3.52

token            plain mean    tf-idf
he                    25.0%     19.3%
was                   25.0%     14.9%
never                 25.0%     30.7%
lost                  25.0%     35.1%


### Sentence and example vectors

In [17]:
def embed(tokens):
    vectors, weights = [], []
    for token in tokens:
        embed_stats["tokens"] += 1
        if token not in wv:
            embed_stats["oov"] += 1
            oov_counts[token] += 1
            continue
        if token not in idf:
            embed_stats["tfidf_unseen"] += 1
        vectors.append(wv[token]) # @todo append token vector
        weights.append(idf.get(token, idf_unseen) if config["sentence_weighting"] == "tfidf" else 1.0) # @todo optionaly if present, weigh it with tfidf
    if not vectors:
        embed_stats["empty_sentences"] += 1
        return np.zeros(wv.vector_size, dtype=np.float32)
    return np.average(vectors, axis=0, weights=weights).astype(np.float32) # @todo apply tfidf with embeddings, and combine the information, what exactly happens here

In [18]:
def embed_example(example):
    vectors = [embed(example[field]) for field in TEXT_FIELDS]
    return np.concatenate(vectors) 

In [19]:
oov_counts = Counter()
embed_stats = Counter()

X_train = np.stack([embed_example(example) for example in train_tokens])
y_train = np.array(train_tokens["target"])
X_valid = np.stack([embed_example(example) for example in valid_tokens])
y_valid = np.array(valid_tokens["target"])
X_test = np.stack([embed_example(example) for example in test_tokens])
y_test = np.array(test_tokens["target"])

print(f"input: {X_train.shape[1]} dimensions, {X_train.dtype}")
print(f"OOV: {embed_stats['oov']} of {embed_stats['tokens']} tokens ({embed_stats['oov'] / embed_stats['tokens']:.2%}), "
      f"empty sentences: {embed_stats['empty_sentences']}, in GloVe but not in the train idf: {embed_stats['tfidf_unseen']}")

input: 1200 dimensions, float32
OOV: 1144 of 670395 tokens (0.17%), empty sentences: 7, in GloVe but not in the train idf: 1417


### DataLoaders - Takes care of Training Batching and Loading

In [20]:
L.seed_everything(config["seed"])

# @todo this has to be optimized or at least it has to be documented how its optimized with badge sizing and concurrency, its the one lecture i missed
warnings.filterwarnings("ignore", ".*does not have many workers.*")

def make_loader(X, y, shuffle):
    dataset = TensorDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.long))
    return DataLoader(dataset, batch_size=config["batch_size"], shuffle=shuffle)

train_loader = make_loader(X_train, y_train, shuffle=True)
valid_loader = make_loader(X_valid, y_valid, shuffle=False)
test_loader = make_loader(X_test, y_test, shuffle=False)

Seed set to 42


### LightningModule Model (Derived from libraries.pdf using coding agents in combination with Lightning Documentation)

In [21]:
class MLP(L.LightningModule): # @todo lightning implementation fine here? any alternatives

    def __init__(self, input_dim, hidden_dim, output_dim, learning_rate):
        super().__init__()
        self.save_hyperparameters()
        # 2 layers as the project asks: linear -> relu -> linear. These are the only trainable weights
        self.layers = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim),
        )
        self.loss_fn = nn.CrossEntropyLoss() # @todo correct loss function for probability distribution with softmax?

    def reset_parameters(self):
        for layer in self.layers:
            if isinstance(layer, nn.Linear):
                layer.reset_parameters()

    def forward(self, x):
        return self.layers(x)

     # @todo investigate and evaluate / tune learning rate iteration can be done autonomously here
    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.learning_rate)

    def shared_step(self, batch, split): # @todo this has to be explainable in detailed since its activation loss etc.
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits, y)
        accuracy = (logits.argmax(dim=1) == y).float().mean()
        self.log(f"loss/{split}", loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log(f"accuracy/{split}", accuracy, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    # @todo check that against ML libraries
    def training_step(self, batch, batch_idx):
        return self.shared_step(batch, "train")

    def validation_step(self, batch, batch_idx):
        return self.shared_step(batch, "valid")

    def test_step(self, batch, batch_idx):
        return self.shared_step(batch, "test")

In [22]:
WEIGHTS_DIR = Path(f"../../../../models/project_1/{MODEL_NAME}/weights")

model = MLP(
    input_dim=len(TEXT_FIELDS) * wv.vector_size,
    hidden_dim=config["hidden_dim"],
    output_dim=2,
    learning_rate=config["learning_rate"],
)
model.reset_parameters()
trainable_parameters = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)

early_stopping = EarlyStopping(monitor="loss/valid", mode="min", patience=config["patience"], verbose=True) # @todo check against documentation and understand it
checkpoint = ModelCheckpoint(dirpath=WEIGHTS_DIR, filename="best-{epoch}", monitor="loss/valid", mode="min", save_top_k=1) # @todo same here but this is largely library oriented

trainer = L.Trainer(
    max_epochs=config["max_epochs"],
    accelerator="auto",
    devices=1,
    logger=WandbLogger(),
    callbacks=[early_stopping, checkpoint], # todo ensure it stops when performance degrades
)
trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=valid_loader)

stopped_epoch = early_stopping.stopped_epoch
print(f"early stopping in epoch {stopped_epoch} (0-based), best checkpoint: {Path(checkpoint.best_model_path).name}")

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


trainable parameters: 307,970 (MLP only, GloVe is not part of the model)


/home/cedi/.local/lib/python3.10/site-packages/lightning/pytorch/loggers/wandb.py:400: There is a wandb run already in progress and newly created instances of `WandbLogger` will reuse this run. If this is not desired, call `wandb.finish()` before instantiating `WandbLogger`.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ layers  │ Sequential       │  307 K │ train │     0 │
│ 1 │ loss_fn │ CrossEntropyLoss │      0 │ train │     0 │
└───┴─────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 307 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 307 K                                                                                                
Total estimated model params size (MB): 1.232                                                                      
Modules in train mode: 5                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/home/cedi/.local/lib/python3.10/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, 
LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

Metric loss/valid improved. New best score: 0.679
Metric loss/valid improved by 0.018 >= min_delta = 0.0. New best score: 0.660
Metric loss/valid improved by 0.005 >= min_delta = 0.0. New best score: 0.655
Metric loss/valid improved by 0.011 >= min_delta = 0.0. New best score: 0.644
Monitored metric loss/valid did not improve in the last 5 records. Best score: 0.644. Signaling Trainer to stop.


early stopping in epoch 8 (0-based), best checkpoint: best-epoch=3.ckpt


### Best Checkpoint + Upload Artifacts to Weights and Biases

In [23]:
trainer.validate(model, dataloaders=valid_loader, ckpt_path="best")
trainer.test(model, dataloaders=test_loader, ckpt_path="best")

run.summary.update({
    "accuracy/majority": np.bincount(y_test).max() / len(y_test),
    "best/loss_valid": checkpoint.best_model_score.item(),
    "best/checkpoint": Path(checkpoint.best_model_path).name,
    "early_stopping/stopped_epoch": stopped_epoch,  # read right after fit, see above
    "model/trainable_parameters": trainable_parameters,
    "oov/tokens": embed_stats["tokens"],
    "oov/occurrences": embed_stats["oov"],
    "oov/rate": embed_stats["oov"] / embed_stats["tokens"],
    "oov/unique_types": len(oov_counts),
    "oov/empty_sentences": embed_stats["empty_sentences"],
    "tfidf/vocabulary_size": len(idf),
    "tfidf/unseen_tokens": embed_stats["tfidf_unseen"],
})

Restoring states from the checkpoint path at /home/cedi/01_Projects/nlp-project-01-abductive-reasoning/models/project_1/glove_tfidf_mlp/weights/best-epoch=3.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at /home/cedi/01_Projects/nlp-project-01-abductive-reasoning/models/project_1/glove_tfidf_mlp/weights/best-epoch=3.ckpt


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃      Validate metric      ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│      accuracy/valid       │     0.621999979019165     │
│        loss/valid         │    0.6442962288856506     │
└───────────────────────────┴───────────────────────────┘

Restoring states from the checkpoint path at /home/cedi/01_Projects/nlp-project-01-abductive-reasoning/models/project_1/glove_tfidf_mlp/weights/best-epoch=3.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at /home/cedi/01_Projects/nlp-project-01-abductive-reasoning/models/project_1/glove_tfidf_mlp/weights/best-epoch=3.ckpt


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│       accuracy/test       │    0.5195822715759277     │
│         loss/test         │    0.8014907240867615     │
└───────────────────────────┴───────────────────────────┘

In [24]:
artifact = wandb.Artifact(MODEL_NAME.replace("_", "-"), type="model", metadata={"loss/valid": checkpoint.best_model_score.item()})
artifact.add_file(checkpoint.best_model_path)
run.log_artifact(artifact, aliases=["best"])
wandb.finish()

accuracy/test,▁
accuracy/train,▁▂▃▃▄▅▆▇█
accuracy/valid,▁▅▆▇▆▇█▇▇▇
epoch,▁▁▂▂▃▃▃▃▄▄▅▅▆▆▆▆▇▇██
loss/test,▁
loss/train,█▇▇▇▆▅▄▂▁
loss/valid,▂▂▁▁▂▂▂▆█▁
trainer/global_step,▁▁▂▂▃▃▄▄▄▄▅▅▆▆▇▇████
accuracy/majority,0.50979
accuracy/test,0.51958
accuracy/train,0.92274
